# Avaliar TODOS os modelos -- holdout (Modelos 1 a 6, os 10 cenários de cada)

Roda, sob o **mesmo critério** (holdout de 29-30 documentos nunca vistos em treino, bbox, IoU>=0,30,
regex de assinatura digital + propagação de texto idêntico -- igual aos notebooks `10.extrair_metricas_holdout*.ipynb`
e `Modelo 3 - avaliar holdout.ipynb`), os 10 cenários de hiperparâmetro de **cada uma das 6 rodadas de treino**
do histórico do projeto, e no final monta UMA tabela comparando o melhor cenário de cada Modelo.

Substitui a necessidade de trocar `PASTA_CENARIOS` manualmente em `Modelo 3 - avaliar holdout.ipynb` a cada
rodada -- este notebook já varre o registro de todos os 6 modelos abaixo.

| Modelo | Pasta de checkpoints | Arquitetura | Status |
|---|---|---|---|
| 1 -- 473 reais, único NER | `bertimbau-pii-v1-modelo1-rerun-cenarios` | single_head | pronto (10/10) |
| 2 -- 473 reais + 500 sintéticos, único NER | `bertimbau-pii-v1-modelo2-rerun-cenarios` | single_head | rodando |
| 3 -- 1000 reais + 500 sintéticos, único NER | `bertimbau-pii-v1-cenarios-com-sinteticos` | single_head | pronto (10/10) |
| 4 -- ~999 reais, único NER, SEM dedup (bug reproduzido) | `bertimbau-pii-v1-modelo4-rerun-cenarios` | single_head | **ainda não treinado -- rode `Modelo 4 - rerun.ipynb`** |
| 5 -- ~999 reais, único NER, COM dedup | `bertimbau-pii-v1-cenarios-sem-sinteticos` | single_head | pronto (10/10) |
| 6 -- 4 cabeças + chunking, sem sintético | `bertimbau-4cabecas-pii-v1-cenarios-sem-sinteticos` | 4_cabecas | pronto (10/10) |

**Cache por cenário**: cada pasta de modelo ganha um `_avaliacao_holdout.json` -- rodar de novo pula cenários
já avaliados (mesma lógica de `Modelo 3 - avaliar holdout.ipynb`). Cenários de modelos ainda não treinados
(ex.: Modelo 4 antes de rodar o rerun) são simplesmente ignorados com um aviso, sem quebrar o notebook --
rode de novo depois que o treino terminar para completar a tabela.

**Pré-requisito:** venv geral do projeto (`.venv`, torch/transformers).

In [1]:
import json
import sys
import time
from collections import defaultdict
from pathlib import Path

import torch
import torch.nn as nn
from transformers import AutoModel, AutoModelForTokenClassification, AutoTokenizer

print(f"CUDA disponível: {torch.cuda.is_available()}")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

d:\Arquivos\ProjetosPython\BFC\LocalScanDocument_v3\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


CUDA disponível: True


## Configuração -- registro dos 6 modelos

In [2]:
def encontrar_raiz_projeto(marcador: str = "requirements.txt") -> Path:
    p = Path.cwd()
    for _ in range(6):
        if (p / marcador).exists():
            return p
        p = p.parent
    raise RuntimeError(f"Não encontrei a raiz do projeto ({marcador}) subindo a partir de {Path.cwd()}")

RAIZ = encontrar_raiz_projeto()
sys.path.insert(0, str(RAIZ / "scripts"))
from pos_processamento_ner import (
    extrair_nome_assinatura_digital,
    propagar_entidades_por_texto_identico,
    mesclar_previsoes_por_janela,
)

MODELO_GABARITO = "gemini-2.5-flash"
TIPOS = ["CPF", "RG", "NOME", "ENDERECO"]
IOU_MINIMO_MATCH = 0.30
MAX_LENGTH = 512
LIMIAR_ALINHAMENTO_MINIMO_PCT = 80

PASTA_EXTRACAO_HOLDOUT = RAIZ / "data" / "documento_social" / "teste_holdout" / "extracao"
PASTA_GABARITO_HOLDOUT = RAIZ / "data" / "documento_social" / "teste_holdout" / "gabarito_alinhado" / MODELO_GABARITO

# Registro dos 6 modelos -- "pasta" aponta pra onde o checkpoint (10 cenários) está OU vai
# estar salvo, mesmo que ainda não exista (ex.: Modelo 4, até o usuário rodar "Modelo 4 -
# rerun.ipynb"). O notebook varre esta lista inteira a cada execução.
REGISTRO_MODELOS = [
    {
        "nome": "Modelo 1 (473 reais, único NER)",
        "pasta": RAIZ / "models" / "bertimbau-pii-v1-modelo1-rerun-cenarios",
        "arquitetura": "single_head",
    },
    {
        "nome": "Modelo 2 (473 reais + 500 sintéticos, único NER)",
        "pasta": RAIZ / "models" / "bertimbau-pii-v1-modelo2-rerun-cenarios",
        "arquitetura": "single_head",
    },
    {
        "nome": "Modelo 3 (1000 reais + 500 sintéticos, único NER)",
        "pasta": RAIZ / "models" / "bertimbau-pii-v1-cenarios-com-sinteticos",
        "arquitetura": "single_head",
    },
    {
        "nome": "Modelo 4 (~999 reais, único NER, SEM dedup -- bug reproduzido de propósito)",
        "pasta": RAIZ / "models" / "bertimbau-pii-v1-modelo4-rerun-cenarios",  # rode "Modelo 4 - rerun.ipynb" primeiro
        "arquitetura": "single_head",
    },
    {
        "nome": "Modelo 5 (~999 reais, único NER, COM dedup)",
        "pasta": RAIZ / "models" / "bertimbau-pii-v1-cenarios-sem-sinteticos",
        "arquitetura": "single_head",
    },
    {
        "nome": "Modelo 6 (4 cabeças + chunking, sem sintético)",
        "pasta": RAIZ / "models" / "bertimbau-4cabecas-pii-v1-cenarios-sem-sinteticos",
        "arquitetura": "4_cabecas",
    },
]

for m in REGISTRO_MODELOS:
    print(f"{'[OK]' if m['pasta'].exists() else '[AINDA NAO EXISTE]':<20} {m['nome']:<62} {m['pasta']}")

[OK]                 Modelo 1 (473 reais, único NER)                                d:\Arquivos\ProjetosPython\BFC\LocalScanDocument_v3\models\bertimbau-pii-v1-modelo1-rerun-cenarios
[OK]                 Modelo 2 (473 reais + 500 sintéticos, único NER)               d:\Arquivos\ProjetosPython\BFC\LocalScanDocument_v3\models\bertimbau-pii-v1-modelo2-rerun-cenarios
[OK]                 Modelo 3 (1000 reais + 500 sintéticos, único NER)              d:\Arquivos\ProjetosPython\BFC\LocalScanDocument_v3\models\bertimbau-pii-v1-cenarios-com-sinteticos
[OK]                 Modelo 4 (~999 reais, único NER, SEM dedup -- bug reproduzido de propósito) d:\Arquivos\ProjetosPython\BFC\LocalScanDocument_v3\models\bertimbau-pii-v1-modelo4-rerun-cenarios
[OK]                 Modelo 5 (~999 reais, único NER, COM dedup)                    d:\Arquivos\ProjetosPython\BFC\LocalScanDocument_v3\models\bertimbau-pii-v1-cenarios-sem-sinteticos
[OK]                 Modelo 6 (4 cabeças + chunking, sem sintético)   

## Carrega o gabarito alinhado + extração do holdout (uma vez, reaproveitado por todos os modelos)

In [3]:
def bbox_contem_centro(caixa: list, bbox_token: list) -> bool:
    x0, y0, x1, y1 = caixa
    cx = (bbox_token[0] + bbox_token[2]) / 2
    cy = (bbox_token[1] + bbox_token[3]) / 2
    return (x0 - 1) <= cx <= (x1 + 1) and (y0 - 1) <= cy <= (y1 + 1)


def construir_exemplos_documento(extracao: dict, gabarito: dict) -> list:
    entidades_por_pagina = {}
    for e in gabarito["entities"]:
        if e.get("match_type") == "unmatched":
            continue
        entidades_por_pagina.setdefault(e["page"], []).append(e)
    exemplos = []
    for pagina_dados in extracao["pages"]:
        tokens = pagina_dados["tokens"]
        if not tokens:
            continue
        exemplos.append({
            "document_id": extracao["document_id"],
            "page": pagina_dados["page"],
            "tokens": [t["text"] for t in tokens],
            "tokens_bbox": [t["bbox"] for t in tokens],
        })
    return exemplos


arquivos_gabarito = sorted(f for f in PASTA_GABARITO_HOLDOUT.glob("*.json") if f.name != "_nao_alinhadas.json")
if not arquivos_gabarito:
    raise SystemExit(f"Nenhum gabarito alinhado encontrado em {PASTA_GABARITO_HOLDOUT} -- rode os notebooks 07 e 09 (em notebooks/bert/) antes.")

exemplos_holdout = []
gabarito_por_doc_pagina = {}
for arq_gab in arquivos_gabarito:
    arq_ext = PASTA_EXTRACAO_HOLDOUT / arq_gab.name
    if not arq_ext.exists():
        print(f"[AVISO] Sem extração local correspondente para {arq_gab.name}, pulando.")
        continue
    gabarito = json.loads(arq_gab.read_text(encoding="utf-8"))
    entidades = gabarito.get("entities", [])
    n_unmatched = sum(1 for e in entidades if e.get("match_type") == "unmatched")
    pct_alinhado = 100 * (len(entidades) - n_unmatched) / len(entidades) if entidades else 100
    if entidades and pct_alinhado < LIMIAR_ALINHAMENTO_MINIMO_PCT:
        print(f"[AVISO] {arq_gab.name}: só {pct_alinhado:.0f}% alinhado, abaixo do limiar -- pulando.")
        continue
    extracao = json.loads(arq_ext.read_text(encoding="utf-8"))
    exemplos_holdout.extend(construir_exemplos_documento(extracao, gabarito))
    for e in entidades:
        if e.get("match_type") == "unmatched":
            continue
        gabarito_por_doc_pagina.setdefault((gabarito["document_id"], e["page"]), []).append(e)

print(f"Documentos de holdout com gabarito utilizável: {len({ex['document_id'] for ex in exemplos_holdout})} | páginas: {len(exemplos_holdout)}")

[AVISO] cm1jln7kb006wdiqdck8ievgk-Constituição 12-09-1996.pdf.json: só 79% alinhado, abaixo do limiar -- pulando.
Documentos de holdout com gabarito utilizável: 29 | páginas: 238


## Funções compartilhadas: bbox/IoU, pós-processamento e métricas (agnósticas à arquitetura)

In [4]:
def agrupar_bboxes_por_linha(tokens_bbox: list, indices: list) -> list:
    if not indices:
        return []
    grupos = []
    grupo_atual = [tokens_bbox[indices[0]]]
    y_ref = tokens_bbox[indices[0]][1]
    for i in indices[1:]:
        bbox = tokens_bbox[i]
        altura_ref = grupo_atual[-1][3] - grupo_atual[-1][1]
        if abs(bbox[1] - y_ref) > max(altura_ref * 0.6, 3):
            grupos.append(grupo_atual)
            grupo_atual = [bbox]
            y_ref = bbox[1]
        else:
            grupo_atual.append(bbox)
    grupos.append(grupo_atual)
    return [[min(b[0] for b in g), min(b[1] for b in g), max(b[2] for b in g), max(b[3] for b in g)] for g in grupos]


def decodificar_bio(tags_previstas: list, tokens_bbox: list, tokens: list) -> list:
    """Decodifica tags BIO de um único vetor de classes (arquitetura single_head)."""
    entidades = []
    i, n = 0, len(tags_previstas)
    while i < n:
        tag = tags_previstas[i]
        if tag == "O":
            i += 1
            continue
        tipo = tag.split("-", 1)[1]
        indices = [i]
        j = i + 1
        while j < n and tags_previstas[j] == f"I-{tipo}":
            indices.append(j)
            j += 1
        entidades.append({
            "type": tipo,
            "bbox": agrupar_bboxes_por_linha(tokens_bbox, indices),
            "indices": indices,
            "text": " ".join(tokens[k] for k in indices),
        })
        i = j
    return entidades


def bbox_externo(lista_bboxes: list):
    if not lista_bboxes:
        return None
    return [min(b[0] for b in lista_bboxes), min(b[1] for b in lista_bboxes),
            max(b[2] for b in lista_bboxes), max(b[3] for b in lista_bboxes)]


def iou(a, b) -> float:
    if a is None or b is None:
        return 0.0
    xi0, yi0 = max(a[0], b[0]), max(a[1], b[1])
    xi1, yi1 = min(a[2], b[2]), min(a[3], b[3])
    inter = max(0, xi1 - xi0) * max(0, yi1 - yi0)
    if inter == 0:
        return 0.0
    area_a = max(0, a[2] - a[0]) * max(0, a[3] - a[1])
    area_b = max(0, b[2] - b[0]) * max(0, b[3] - b[1])
    uniao = area_a + area_b - inter
    return inter / uniao if uniao > 0 else 0.0


def avaliar_com_pos_processamento(predicoes_por_pagina: dict, exemplos: list) -> dict:
    """Base + regex de assinatura digital + propagação de texto idêntico -- pipeline completo
    padrão do projeto, agnóstico à arquitetura (só consome a lista de entidades já decodificada
    por página, não importa como o modelo por trás gerou essa lista)."""
    contadores = {t: {"tp": 0, "fp": 0, "fn": 0} for t in TIPOS}
    exemplos_por_doc = defaultdict(list)
    for ex in exemplos:
        exemplos_por_doc[ex["document_id"]].append(ex)

    for doc_id, exs in exemplos_por_doc.items():
        previstas_por_pagina = {}
        entidades_documento = []
        for ex in exs:
            previstas = [dict(p) for p in predicoes_por_pagina[(doc_id, ex["page"])]]
            extras = extrair_nome_assinatura_digital(ex["tokens"], ex["tokens_bbox"])
            for extra in extras:
                bbox_extra = bbox_externo(extra["bbox"])
                ja_coberto = any(
                    p["type"] == "NOME" and iou(bbox_extra, bbox_externo(p["bbox"])) > 0.1
                    for p in previstas
                )
                if not ja_coberto:
                    previstas.append(extra)
            for p in previstas:
                p["page"] = ex["page"]
            previstas_por_pagina[ex["page"]] = previstas
            entidades_documento.extend(previstas)

        paginas_doc = [
            {"page": ex["page"], "tokens": [{"text": t, "bbox": b} for t, b in zip(ex["tokens"], ex["tokens_bbox"])]}
            for ex in exs
        ]
        entidades_documento = propagar_entidades_por_texto_identico(
            entidades_documento, paginas_doc, tipos_propagaveis=tuple(TIPOS)
        )
        previstas_por_pagina = defaultdict(list)
        for p in entidades_documento:
            previstas_por_pagina[p["page"]].append(p)

        for ex in exs:
            previstas = previstas_por_pagina.get(ex["page"], [])
            gold = gabarito_por_doc_pagina.get((doc_id, ex["page"]), [])
            previstas_por_tipo = {}
            for p in previstas:
                previstas_por_tipo.setdefault(p["type"], []).append(p)

            usadas = set()
            for g in gold:
                tipo = g["type"]
                candidatos = previstas_por_tipo.get(tipo, [])
                bbox_gold = bbox_externo(g.get("bbox") or [])
                melhor_iou, melhor_idx = 0.0, None
                for i, p in enumerate(candidatos):
                    if id(p) in usadas:
                        continue
                    score = iou(bbox_gold, bbox_externo(p["bbox"]))
                    if score > melhor_iou:
                        melhor_iou, melhor_idx = score, i
                if melhor_idx is not None and melhor_iou >= IOU_MINIMO_MATCH:
                    usadas.add(id(candidatos[melhor_idx]))
                    contadores[tipo]["tp"] += 1
                else:
                    contadores[tipo]["fn"] += 1

            for tipo, candidatos in previstas_por_tipo.items():
                for p in candidatos:
                    if id(p) not in usadas:
                        contadores.setdefault(tipo, {"tp": 0, "fp": 0, "fn": 0})
                        contadores[tipo]["fp"] += 1
    return contadores


def metricas_geral(contadores: dict) -> dict:
    tp_g = sum(c["tp"] for c in contadores.values())
    fp_g = sum(c["fp"] for c in contadores.values())
    fn_g = sum(c["fn"] for c in contadores.values())
    precision = tp_g / (tp_g + fp_g) if (tp_g + fp_g) else 0.0
    recall = tp_g / (tp_g + fn_g) if (tp_g + fn_g) else 0.0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) else 0.0
    return {"tp": tp_g, "fp": fp_g, "fn": fn_g, "precision": precision, "recall": recall, "f1": f1}


def metricas_por_tipo(contadores: dict) -> dict:
    resultado = {}
    for tipo in TIPOS:
        c = contadores.get(tipo, {"tp": 0, "fp": 0, "fn": 0})
        tp, fp, fn = c["tp"], c["fp"], c["fn"]
        precision = tp / (tp + fp) if (tp + fp) else 0.0
        recall = tp / (tp + fn) if (tp + fn) else 0.0
        f1 = 2 * precision * recall / (precision + recall) if (precision + recall) else 0.0
        resultado[tipo] = {"tp": tp, "fp": fp, "fn": fn, "precision": precision, "recall": recall, "f1": f1}
    return resultado

## Avaliador -- arquitetura `single_head` (um vetor de 9 classes, `AutoModelForTokenClassification`)

In [5]:
def avaliar_cenario_single_head(pasta_modelo: Path) -> dict:
    tokenizer = AutoTokenizer.from_pretrained(pasta_modelo)
    model = AutoModelForTokenClassification.from_pretrained(pasta_modelo).to(device)
    model.eval()

    label_list = list(model.config.id2label.values())
    id2label = {i: l for i, l in enumerate(label_list)}

    @torch.no_grad()
    def _prever_tags_janela(tokens_janela: list) -> list:
        encoding = tokenizer(tokens_janela, is_split_into_words=True, truncation=True, max_length=MAX_LENGTH, return_tensors="pt")
        encoding = {k: v.to(device) for k, v in encoding.items() if k in ("input_ids", "attention_mask", "token_type_ids")}
        logits = model(**encoding).logits[0]
        pred_ids = logits.argmax(dim=-1).tolist()
        word_ids = tokenizer(tokens_janela, is_split_into_words=True, truncation=True, max_length=MAX_LENGTH).word_ids()
        tags = ["O"] * len(tokens_janela)
        visto = set()
        for pid, wid in zip(pred_ids, word_ids):
            if wid is None or wid in visto:
                continue
            visto.add(wid)
            tags[wid] = id2label[pid]
        return tags

    def prever_pagina(tokens: list, tokens_bbox: list) -> list:
        if not tokens:
            return []
        tags = mesclar_previsoes_por_janela(tokens, tokenizer, _prever_tags_janela, max_length=MAX_LENGTH)
        return decodificar_bio(tags, tokens_bbox, tokens)

    predicoes_por_pagina = {}
    for ex in exemplos_holdout:
        predicoes_por_pagina[(ex["document_id"], ex["page"])] = prever_pagina(ex["tokens"], ex["tokens_bbox"])

    contadores = avaliar_com_pos_processamento(predicoes_por_pagina, exemplos_holdout)

    del model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return {"geral": metricas_geral(contadores), "por_tipo": metricas_por_tipo(contadores)}

## Avaliador -- arquitetura `4_cabecas` (encoder compartilhado + 1 cabeça linear por tipo)

In [6]:
class BERTimbau4Cabecas(nn.Module):
    """Mesma definição de notebooks/bert_4_cabecas/06 e 10 -- reproduzida aqui por
    autocontenção (cada notebook do projeto é executável isoladamente)."""

    def __init__(self, checkpoint_base: str, tipos: list):
        super().__init__()
        self.tipos = list(tipos)
        self.encoder = AutoModel.from_pretrained(checkpoint_base)
        hidden = self.encoder.config.hidden_size
        p_dropout = getattr(self.encoder.config, "hidden_dropout_prob", 0.1)
        self.dropout = nn.Dropout(p_dropout)
        self.cabecas = nn.ModuleDict({tipo: nn.Linear(hidden, 3) for tipo in self.tipos})

    def forward(self, input_ids, attention_mask, labels_por_tipo: dict = None):
        saida_encoder = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        oculto = self.dropout(saida_encoder.last_hidden_state)
        logits_por_tipo = {tipo: self.cabecas[tipo](oculto) for tipo in self.tipos}
        return {"logits_por_tipo": logits_por_tipo}

    @classmethod
    def load(cls, pasta, device="cpu"):
        pasta = Path(pasta)
        config = json.loads((pasta / "config_4cabecas.json").read_text(encoding="utf-8"))
        modelo = cls(str(pasta / "encoder"), config["tipos"])
        cabecas_state = torch.load(pasta / "cabecas.pt", map_location=device)
        for tipo in config["tipos"]:
            modelo.cabecas[tipo].load_state_dict(cabecas_state[tipo])
        return modelo.to(device)


_LABEL2ID_POR_TIPO = {tipo: {"O": 0, f"B-{tipo}": 1, f"I-{tipo}": 2} for tipo in TIPOS}
_ID2LABEL_POR_TIPO = {tipo: {v: k for k, v in _LABEL2ID_POR_TIPO[tipo].items()} for tipo in TIPOS}


def avaliar_cenario_4_cabecas(pasta_modelo: Path) -> dict:
    tokenizer = AutoTokenizer.from_pretrained(pasta_modelo / "encoder")
    model = BERTimbau4Cabecas.load(pasta_modelo, device=device)
    model.eval()

    @torch.no_grad()
    def _prever_tags_janela_tipo(tokens_janela: list, tipo: str) -> list:
        encoding = tokenizer(tokens_janela, is_split_into_words=True, truncation=True, max_length=MAX_LENGTH, return_tensors="pt")
        encoding = {k: v.to(device) for k, v in encoding.items() if k != "token_type_ids"}
        saida = model(**encoding)
        pred_ids = saida["logits_por_tipo"][tipo][0].argmax(dim=-1).tolist()
        word_ids = tokenizer(tokens_janela, is_split_into_words=True, truncation=True, max_length=MAX_LENGTH).word_ids()
        tags_por_palavra = ["O"] * len(tokens_janela)
        visto = set()
        for pid, word_idx in zip(pred_ids, word_ids):
            if word_idx is None or word_idx in visto:
                continue
            visto.add(word_idx)
            tags_por_palavra[word_idx] = _ID2LABEL_POR_TIPO[tipo][pid]
        return tags_por_palavra

    def prever_pagina_4cabecas(tokens: list, tokens_bbox: list) -> list:
        if not tokens:
            return []
        entidades = []
        for tipo in TIPOS:
            tags_por_palavra = mesclar_previsoes_por_janela(
                tokens, tokenizer, lambda tj, t=tipo: _prever_tags_janela_tipo(tj, t), max_length=MAX_LENGTH
            )
            i, n = 0, len(tags_por_palavra)
            while i < n:
                if tags_por_palavra[i] == "O":
                    i += 1
                    continue
                indices = [i]
                j = i + 1
                while j < n and tags_por_palavra[j] == f"I-{tipo}":
                    indices.append(j)
                    j += 1
                entidades.append({
                    "type": tipo,
                    "bbox": agrupar_bboxes_por_linha(tokens_bbox, indices),
                    "indices": indices,
                    "text": " ".join(tokens[k] for k in indices),
                })
                i = j
        return entidades

    predicoes_por_pagina = {}
    for ex in exemplos_holdout:
        predicoes_por_pagina[(ex["document_id"], ex["page"])] = prever_pagina_4cabecas(ex["tokens"], ex["tokens_bbox"])

    contadores = avaliar_com_pos_processamento(predicoes_por_pagina, exemplos_holdout)

    del model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return {"geral": metricas_geral(contadores), "por_tipo": metricas_por_tipo(contadores)}

## Roda os 6 modelos (10 cenários cada), com cache por cenário em `_avaliacao_holdout.json`

In [7]:
_AVALIADOR_POR_ARQUITETURA = {
    "single_head": avaliar_cenario_single_head,
    "4_cabecas": avaliar_cenario_4_cabecas,
}
_ARQUIVO_CONFIG_POR_ARQUITETURA = {
    "single_head": "config.json",
    "4_cabecas": "config_4cabecas.json",
}

resultados_por_modelo = {}

for modelo in REGISTRO_MODELOS:
    nome, pasta, arquitetura = modelo["nome"], modelo["pasta"], modelo["arquitetura"]
    print(f"\n{'=' * 100}\n{nome}\n{'=' * 100}")

    if not pasta.exists():
        print(f"[AVISO] Pasta ainda não existe -- {pasta} -- pulando este modelo por enquanto.")
        continue

    arquivo_config = _ARQUIVO_CONFIG_POR_ARQUITETURA[arquitetura]
    nomes_cenarios = sorted(p.name for p in pasta.iterdir() if p.is_dir() and (p / arquivo_config).exists())
    if not nomes_cenarios:
        print(f"[AVISO] Nenhum cenário completo encontrado em {pasta} (procurando '{arquivo_config}') -- pulando.")
        continue
    print(f"{len(nomes_cenarios)} cenário(s) encontrado(s): {nomes_cenarios}")

    arquivo_saida = pasta / "_avaliacao_holdout.json"
    resultados = {}
    if arquivo_saida.exists():
        resultados = json.loads(arquivo_saida.read_text(encoding="utf-8"))
        print(f"Reaproveitando {len(resultados)} cenário(s) já avaliado(s) anteriormente.")

    avaliar_cenario = _AVALIADOR_POR_ARQUITETURA[arquitetura]
    for nome_cenario in nomes_cenarios:
        if nome_cenario in resultados:
            print(f"  {nome_cenario}: já avaliado, pulando.")
            continue
        inicio = time.time()
        resultado = avaliar_cenario(pasta / nome_cenario)
        duracao = time.time() - inicio
        g = resultado["geral"]
        print(f"  {nome_cenario}: F1={g['f1']:.4f}  P={g['precision']:.4f}  R={g['recall']:.4f}  ({duracao:.0f}s)")
        resultados[nome_cenario] = resultado
        arquivo_saida.write_text(json.dumps(resultados, ensure_ascii=False, indent=2), encoding="utf-8")

    resultados_por_modelo[nome] = resultados
    print(f"Resultados salvos em: {arquivo_saida}")


Modelo 1 (473 reais, único NER)
10 cenário(s) encontrado(s): ['c01_baseline', 'c02_lr_baixo', 'c03_lr_alto', 'c04_batch8', 'c05_batch16', 'c06_sem_weight_decay', 'c07_weight_decay_alto', 'c08_warmup', 'c09_lrbaixo_batch8_warmup', 'c10_lralto_batch16_wdalto_warmup']
Reaproveitando 10 cenário(s) já avaliado(s) anteriormente.
  c01_baseline: já avaliado, pulando.
  c02_lr_baixo: já avaliado, pulando.
  c03_lr_alto: já avaliado, pulando.
  c04_batch8: já avaliado, pulando.
  c05_batch16: já avaliado, pulando.
  c06_sem_weight_decay: já avaliado, pulando.
  c07_weight_decay_alto: já avaliado, pulando.
  c08_warmup: já avaliado, pulando.
  c09_lrbaixo_batch8_warmup: já avaliado, pulando.
  c10_lralto_batch16_wdalto_warmup: já avaliado, pulando.
Resultados salvos em: d:\Arquivos\ProjetosPython\BFC\LocalScanDocument_v3\models\bertimbau-pii-v1-modelo1-rerun-cenarios\_avaliacao_holdout.json

Modelo 2 (473 reais + 500 sintéticos, único NER)
6 cenário(s) encontrado(s): ['c01_baseline', 'c02_lr_ba

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 8057.25it/s]


  c01_baseline: F1=0.8656  P=0.7963  R=0.9482  (65s)


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 376.56it/s]


  c02_lr_baixo: F1=0.8573  P=0.7829  R=0.9473  (67s)


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 425.38it/s]


... (11 outputs left out)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 404.87it/s]


  c09_lrbaixo_batch8_warmup: F1=0.8650  P=0.7990  R=0.9429  (65s)


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 392.10it/s]


  c10_lralto_batch16_wdalto_warmup: F1=0.8218  P=0.7374  R=0.9279  (65s)
Resultados salvos em: d:\Arquivos\ProjetosPython\BFC\LocalScanDocument_v3\models\bertimbau-4cabecas-pii-v1-cenarios-sem-sinteticos\_avaliacao_holdout.json


## Resumo final -- melhor cenário de cada modelo, lado a lado

In [8]:
print(f"{'Modelo':<62}{'Melhor cenário':<22}{'F1':<8}{'Precision':<12}{'Recall':<8}")
print("-" * 112)
for modelo in REGISTRO_MODELOS:
    nome = modelo["nome"]
    resultados = resultados_por_modelo.get(nome)
    if not resultados:
        print(f"{nome:<62}{'(sem resultado)':<22}")
        continue
    melhor_nome, melhor = max(resultados.items(), key=lambda kv: kv[1]["geral"]["f1"])
    g = melhor["geral"]
    print(f"{nome:<62}{melhor_nome:<22}{g['f1']:<8.4f}{g['precision']:<12.4f}{g['recall']:<8.4f}")

print("\nDetalhe por tipo de entidade do melhor cenário de cada modelo:")
for modelo in REGISTRO_MODELOS:
    nome = modelo["nome"]
    resultados = resultados_por_modelo.get(nome)
    if not resultados:
        continue
    melhor_nome, melhor = max(resultados.items(), key=lambda kv: kv[1]["geral"]["f1"])
    print(f"\n{nome} -- {melhor_nome}")
    for tipo in TIPOS:
        m = melhor["por_tipo"][tipo]
        print(f"    {tipo:10s} F1={m['f1']:.3f}  P={m['precision']:.3f}  R={m['recall']:.3f}")

Modelo                                                        Melhor cenário        F1      Precision   Recall  
----------------------------------------------------------------------------------------------------------------
Modelo 1 (473 reais, único NER)                               c08_warmup            0.8509  0.7735      0.9455  
Modelo 2 (473 reais + 500 sintéticos, único NER)              c03_lr_alto           0.8543  0.8014      0.9148  
Modelo 3 (1000 reais + 500 sintéticos, único NER)             c02_lr_baixo          0.8680  0.8009      0.9473  
Modelo 4 (~999 reais, único NER, SEM dedup -- bug reproduzido de propósito)c01_baseline          0.7832  0.6798      0.9236  
Modelo 5 (~999 reais, único NER, COM dedup)                   c08_warmup            0.8665  0.8028      0.9411  
Modelo 6 (4 cabeças + chunking, sem sintético)                c06_sem_weight_decay  0.8730  0.8249      0.9271  

Detalhe por tipo de entidade do melhor cenário de cada modelo:

Modelo 1 (473 reai